In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier


In [9]:
# Tablo 1: Demografik Bilgiler
df_musteri = pd.DataFrame({
    'Musteri_ID': [101, 102, 103, 104, 105, 106, 107, 108, 109, 110],
    'Yasi': ['25', '45', '35', '50', '23', '40', '60', '20', '30', '55'],  # str tipinde
    'Sehir': ['Istanbul', 'Ankara', 'Istanbul', 'Izmir', 'Ankara', 'Istanbul', 'Izmir', 'Ankara', 'Istanbul', 'Izmir']
})

# Tablo 2: Finansal Durum
df_finans = pd.DataFrame({
    'Musteri_ID': [101, 102, 103, 104, 105, 106, 107, 108, 109, 110],
    'Gelir': [25000.0, np.nan, 45000.0, 18000.0, np.nan, 55000.0, 12000.0, 22000.0, 60000.0, 15000.0],
    'Kredi_Skoru': [550, 720, 680, 500, 580, 790, 480, 610, 810, 520],
    'Borclu_Mu': [1, 0, 0, 1, 1, 0, 1, 0, 0, 1]  # Target (1: Kredisini Ödemedi/Riskli, 0: Ödedi/Temiz)
})

In [10]:
df=pd.merge(df_musteri,df_finans,on="Musteri_ID")
df

,Musteri_ID,Yasi,Sehir,Gelir,Kredi_Skoru,Borclu_Mu
0,101,25,Istanbul,25000.0,550,1
1,102,45,Ankara,NaN,720,0
2,103,35,Istanbul,45000.0,680,0
3,104,50,Izmir,18000.0,500,1
4,105,23,Ankara,NaN,580,1
5,106,40,Istanbul,55000.0,790,0
6,107,60,Izmir,12000.0,480,1
7,108,20,Ankara,22000.0,610,0
8,109,30,Istanbul,60000.0,810,0
9,110,55,Izmir,15000.0,520,1


In [11]:
df["Yasi"]=df["Yasi"].astype(int)
df["Yasi"]

0    25
1    45
2    35
3    50
4    23
5    40
6    60
7    20
8    30
9    55
Name: Yasi, dtype: int64

In [12]:
sehir_ort=df.groupby("Sehir")["Gelir"].transform("mean")
df["Gelir"]=df["Gelir"].fillna(sehir_ort)
df["Gelir"]

0    25000.0
1    22000.0
2    45000.0
3    18000.0
4    22000.0
5    55000.0
6    12000.0
7    22000.0
8    60000.0
9    15000.0
Name: Gelir, dtype: float64

In [13]:
df["Yuksek_Risk_Sinyali"]=np.where((df["Gelir"]<25000)&(df["Kredi_Skoru"]<600),1,0)
df=pd.get_dummies(df,columns=["Sehir"],drop_first=True,dtype=int)
df

,Musteri_ID,Yasi,Gelir,Kredi_Skoru,Borclu_Mu,Yuksek_Risk_Sinyali,Sehir_Istanbul,Sehir_Izmir
0,101,25,25000.0,550,1,0,1,0
1,102,45,22000.0,720,0,0,0,0
2,103,35,45000.0,680,0,0,1,0
3,104,50,18000.0,500,1,1,0,1
4,105,23,22000.0,580,1,1,0,0
5,106,40,55000.0,790,0,0,1,0
6,107,60,12000.0,480,1,1,0,1
7,108,20,22000.0,610,0,0,0,0
8,109,30,60000.0,810,0,0,1,0
9,110,55,15000.0,520,1,1,0,1


In [14]:
y=df["Borclu_Mu"]
x=df.drop(["Borclu_Mu","Musteri_ID"],axis=1)


In [15]:

x_train,x_test,y_train,y_test=train_test_split(x,y,random_state=42,train_size=0.8)
rf=RandomForestClassifier()
model=rf.fit(x_train,y_train)
model.score(x_test,y_test)

1.0

In [18]:
def kredi_modeli_egit(df:pd.DataFrame)->float:
    y=df["Borclu_Mu"]
    x=df.drop("Borclu_Mu",axis=1)
    
    x_train,x_test,y_train,y_test=train_test_split(x,y,random_state=42,train_size=0.8)
    rf=RandomForestClassifier()
    model=rf.fit(x_train,y_train)
    return model.score(x_test,y_test)


In [19]:
kredi_modeli_egit(df)

1.0